## Modelo final con Corrección de Coeficientes (Prais Winsten - ARIMAX(1,0,0))

In [86]:
#pip install pandas numpy scipy scikit-learn statsmodels

In [87]:
# Librerías para el análisis estadístico y de series de tiempo
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.regression.linear_model import GLSAR
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.stats.diagnostic import het_breuschpagan, acorr_ljungbox, het_arch
from statsmodels.stats.stattools import jarque_bera, durbin_watson
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tsa.stattools import adfuller
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
import warnings
warnings.filterwarnings('ignore')

In [88]:
# Carga de datos y función para construir el dataset mensual por estación
DATA = '../data_estaciones'
ndvi_mensual = pd.read_csv(f'{DATA}/ndvi_mensual_estaciones_2020_2025.csv', parse_dates=['mes'])
evi = pd.read_csv(f'{DATA}/EVI_estaciones.csv', parse_dates=['fecha'])

def construir(estacion, indice):
    df = pd.read_csv(f'{DATA}/BD_{estacion}_limpia.csv', parse_dates=['date'])
    df = df[(df.date >= '2021-01-01') & (df.date <= '2025-12-31')]
    g = df.set_index('date')
    pm10 = g['PM10'].resample('MS').median()
    rh = g['RH'].resample('MS').mean()
    mensual = pd.DataFrame({'mes': pm10.index, 'PM10': pm10.values, 'RH': rh.values})
    if indice == 'ndvi':
        ind = ndvi_mensual[ndvi_mensual.estacion == estacion][['mes', 'ndvi_mediana']]
        ind = ind.rename(columns={'ndvi_mediana': 'indice_veg'})
    else:
        e = evi[evi.estacion == estacion].copy()
        e['mes'] = e.fecha.dt.to_period('M').dt.to_timestamp()
        ind = e.groupby('mes')['EVI'].mean().reset_index().rename(columns={'EVI': 'indice_veg'})
    d = mensual.merge(ind, on='mes').dropna().sort_values('mes').reset_index(drop=True)
    d['mesnum'] = d.mes.dt.month
    return d

In [89]:
# Verificación de supuestos (por estación e índice)
print(f"{'idx':5s}{'est':4s}{'n':4s}{'VIF_ind':9s}{'VIF_RH':8s}{'BP_p':8s}{'LB_p':8s}{'DW':6s}{'JB_p':8s}")
for indice in ['ndvi', 'evi']:
    for est in ['NE2', 'NE3']:
        d = construir(est, indice)
        dummies = pd.get_dummies(d.mesnum, prefix='m', drop_first=True).astype(float)
        X = pd.concat([d[['indice_veg', 'RH']].reset_index(drop=True), dummies.reset_index(drop=True)], axis=1)
        Xc = sm.add_constant(X)

        vif_indice = variance_inflation_factor(X.values, X.columns.get_loc('indice_veg'))
        vif_rh = variance_inflation_factor(X.values, X.columns.get_loc('RH'))

        modelo_ols = sm.OLS(d['PM10'], Xc).fit()

        bp_stat, bp_p, _, _ = het_breuschpagan(modelo_ols.resid, Xc)

        lb = acorr_ljungbox(modelo_ols.resid, lags=[6], return_df=True)
        dw = durbin_watson(modelo_ols.resid)

        jb_stat, jb_p, skew, kurt = jarque_bera(modelo_ols.resid)

        print(f"{indice:5s}{est:4s}{len(d):<4d}{vif_indice:<9.2f}{vif_rh:<8.2f}{bp_p:<8.4f}{lb.lb_pvalue.iloc[0]:<8.4f}{dw:<6.2f}{jb_p:<8.4f}")

idx  est n   VIF_ind  VIF_RH  BP_p    LB_p    DW    JB_p    
ndvi NE2 60  1.24     1.73    0.1293  0.0000  0.54  0.3996  
ndvi NE3 60  1.73     2.11    0.2536  0.0002  1.28  0.4011  
evi  NE2 60  1.72     2.00    0.2778  0.0000  0.58  0.2125  
evi  NE3 60  3.49     2.88    0.1368  0.0007  1.35  0.3883  


#### Modelo con corrección HAC

In [90]:
# Modelo con corrección HAC 
resultados = []
for indice in ['ndvi', 'evi']:
    for est in ['NE2', 'NE3']:
        d = construir(est, indice)
        dummies = pd.get_dummies(d.mesnum, prefix='m', drop_first=True).astype(float)
        X_rh = sm.add_constant(pd.concat([d[['indice_veg']].reset_index(drop=True), dummies.reset_index(drop=True)], axis=1))
        m_rh = sm.OLS(d.RH, X_rh).fit(cov_type='HAC', cov_kwds={'maxlags': 4})
        X_pm = sm.add_constant(pd.concat([d[['indice_veg', 'RH']].reset_index(drop=True), dummies.reset_index(drop=True)], axis=1))
        m_pm = sm.OLS(d.PM10, X_pm).fit(cov_type='HAC', cov_kwds={'maxlags': 4})
        resultados.append({
            'indice': indice.upper(), 'estacion': est, 'n': len(d),
            'indice_RH_coef': round(m_rh.params['indice_veg'], 2), 'indice_RH_p': round(m_rh.pvalues['indice_veg'], 4),
            'RH_PM10_coef': round(m_pm.params['RH'], 3), 'RH_PM10_p': round(m_pm.pvalues['RH'], 4),
            'indice_PM10_coef': round(m_pm.params['indice_veg'], 2), 'indice_PM10_p': round(m_pm.pvalues['indice_veg'], 4),
            'R2': round(m_pm.rsquared, 3)
        })
print(pd.DataFrame(resultados).to_string(index=False))

indice estacion  n  indice_RH_coef  indice_RH_p  RH_PM10_coef  RH_PM10_p  indice_PM10_coef  indice_PM10_p    R2
  NDVI      NE2 60           -3.28       0.9554        -0.567     0.0354            197.05         0.0792 0.458
  NDVI      NE3 60            5.08       0.4156        -0.471     0.0000              6.61         0.4760 0.531
   EVI      NE2 60          108.23       0.0016        -0.805     0.0002            182.58         0.0163 0.458
   EVI      NE3 60           63.05       0.0001        -0.609     0.0006             33.99         0.1414 0.549


In [91]:
# Función rmse_cv(): error de predicción por validación cruzada
def rmse_cv(d, incluir_indice_rh=True, k=5, seed=42):
    dummies = pd.get_dummies(d.mesnum, prefix='m', drop_first=True).astype(float)
    if incluir_indice_rh:
        X = pd.concat([d[['indice_veg', 'RH']].reset_index(drop=True), dummies.reset_index(drop=True)], axis=1)
    else:
        X = dummies.reset_index(drop=True)
    X = sm.add_constant(X).values
    y = d['PM10'].values
    kf = KFold(n_splits=k, shuffle=True, random_state=seed)
    errores = []
    for train_idx, test_idx in kf.split(X):
        modelo = sm.OLS(y[train_idx], X[train_idx]).fit()
        pred = modelo.predict(X[test_idx])
        errores.append(np.sqrt(mean_squared_error(y[test_idx], pred)))
    return np.mean(errores)

In [92]:
# Validación: ¿el modelo generaliza fuera de muestra?
print(f"\n{'idx':5s}{'est':4s}{'RMSE solo-mes':15s}{'RMSE completo':15s}{'Mejora %':10s}")
for indice in ['ndvi', 'evi']:
    for est in ['NE2', 'NE3']:
        d = construir(est, indice)
        rmse_base = rmse_cv(d, incluir_indice_rh=False)
        rmse_completo = rmse_cv(d, incluir_indice_rh=True)
        mejora = 100 * (rmse_base - rmse_completo) / rmse_base
        print(f"{indice:5s}{est:4s}{rmse_base:<15.2f}{rmse_completo:<15.2f}{mejora:<10.1f}")


idx  est RMSE solo-mes  RMSE completo  Mejora %  
ndvi NE2 11.66          12.17          -4.4      
ndvi NE3 6.87           6.56           4.5       
evi  NE2 11.66          11.52          1.3       
evi  NE3 6.87           6.83           0.6       


#### Ajustes con los tres modelos (OLS + HAC, Prais-Winsten, ARIMAX(1,0,0))

In [115]:
# Función ajustar_tres_metodos(): ajusta OLS+HAC, Prais-Winsten y ARIMAX
def ajustar_tres_metodos(y, exog_df, maxlags_hac=4, maxiter_arimax=500):
    """Ajusta y~exog con OLS+HAC, Prais-Winsten y ARIMAX(1,0,0); regresa dict de resultados."""
    Xc = sm.add_constant(exog_df)

    m_hac = sm.OLS(y, Xc).fit(cov_type='HAC', cov_kwds={'maxlags': maxlags_hac})
    m_pw = GLSAR(y, Xc, rho=1).iterative_fit(maxiter=20)
    m_arimax = ARIMA(y, order=(1, 0, 0), exog=exog_df).fit(method_kwargs={'maxiter': maxiter_arimax})

    resid_arimax = m_arimax.resid
    lb_p = acorr_ljungbox(resid_arimax, lags=[12], return_df=True).lb_pvalue.iloc[0]
    jb_p = jarque_bera(resid_arimax)[1]

    return {'hac': m_hac, 'pw': m_pw, 'arimax': m_arimax, 'lb_p_arimax': lb_p, 'jb_p_arimax': jb_p}

In [113]:
# Ruta indirecta completa: a (índice→RH) y b (RH→PM10) bajo tres estimadores
resultados_indirecto = []
for indice in ['ndvi', 'evi']:
    for est in ['NE2', 'NE3']:
        d = construir(est, indice)
        dummies = pd.get_dummies(d.mesnum, prefix='m', drop_first=True).astype(float)

        exog_a = pd.concat([d[['indice_veg']].reset_index(drop=True), dummies.reset_index(drop=True)], axis=1)
        modelos_a = ajustar_tres_metodos(d['RH'].astype(float), exog_a)

        exog_b = pd.concat([d[['indice_veg', 'RH']].reset_index(drop=True), dummies.reset_index(drop=True)], axis=1)
        modelos_b = ajustar_tres_metodos(d['PM10'].astype(float), exog_b)

        fila = {'indice': indice.upper(), 'estacion': est, 'n': len(d)}
        for metodo in ['hac', 'pw', 'arimax']:
            a = modelos_a[metodo].params['indice_veg']
            a_p = modelos_a[metodo].pvalues['indice_veg']
            b = modelos_b[metodo].params['RH']
            b_p = modelos_b[metodo].pvalues['RH']
            indirecto = a * b
            fila[f'a_{metodo}'] = round(a, 3)
            fila[f'a_p_{metodo}'] = round(a_p, 4)
            fila[f'b_{metodo}'] = round(b, 3)
            fila[f'b_p_{metodo}'] = round(b_p, 4)
            fila[f'indirecto_{metodo}'] = round(indirecto, 4)
        fila['LjungBox_p_a_arimax'] = round(modelos_a['lb_p_arimax'], 4)
        fila['LjungBox_p_b_arimax'] = round(modelos_b['lb_p_arimax'], 4)
        resultados_indirecto.append(fila)

tabla_indirecto = pd.DataFrame(resultados_indirecto)

In [116]:
# Vista resumida de a, b e indirecto por método
for metodo, nombre in [('hac', 'OLS + HAC'), ('pw', 'Prais-Winsten'), ('arimax', 'ARIMAX(1,0,0)')]:
    cols = ['indice', 'estacion', 'n', f'a_{metodo}', f'a_p_{metodo}', f'b_{metodo}', f'b_p_{metodo}', f'indirecto_{metodo}']
    print(f"\n--- {nombre} ---")
    print(tabla_indirecto[cols].to_string(index=False))


--- OLS + HAC ---
indice estacion  n   a_hac  a_p_hac  b_hac  b_p_hac  indirecto_hac
  NDVI      NE2 60  -3.276   0.9554 -0.567   0.0354         1.8583
  NDVI      NE3 60   5.083   0.4156 -0.471   0.0000        -2.3929
   EVI      NE2 60 108.227   0.0016 -0.805   0.0002       -87.1093
   EVI      NE3 60  63.046   0.0001 -0.609   0.0006       -38.4162

--- Prais-Winsten ---
indice estacion  n    a_pw  a_p_pw   b_pw  b_p_pw  indirecto_pw
  NDVI      NE2 60 -13.396  0.8011 -0.675  0.0007        9.0403
  NDVI      NE3 60   1.588  0.8761 -0.623  0.0004       -0.9899
   EVI      NE2 60  97.132  0.0192 -0.741  0.0003      -71.9460
   EVI      NE3 60  58.055  0.0005 -0.698  0.0004      -40.5089

--- ARIMAX(1,0,0) ---
indice estacion  n  a_arimax  a_p_arimax  b_arimax  b_p_arimax  indirecto_arimax
  NDVI      NE2 60   -16.450      0.7335    -0.687      0.0000           11.3040
  NDVI      NE3 60     1.842      0.8872    -0.626      0.0002           -1.1525
   EVI      NE2 60    94.943      0.0

In [96]:
# Modelo con corrección HAC (SOLO corrige inferencia, NO las betas)
resultados_hac = []
for indice in ['ndvi', 'evi']:
    for est in ['NE2', 'NE3']:
        d = construir(est, indice)
        dummies = pd.get_dummies(d.mesnum, prefix='m', drop_first=True).astype(float)
        X_rh = sm.add_constant(pd.concat([d[['indice_veg']].reset_index(drop=True), dummies.reset_index(drop=True)], axis=1))
        m_rh = sm.OLS(d.RH, X_rh).fit(cov_type='HAC', cov_kwds={'maxlags': 4})
        X_pm = sm.add_constant(pd.concat([d[['indice_veg', 'RH']].reset_index(drop=True), dummies.reset_index(drop=True)], axis=1))
        m_pm = sm.OLS(d.PM10, X_pm).fit(cov_type='HAC', cov_kwds={'maxlags': 4})
        resultados_hac.append({
            'indice': indice.upper(), 'estacion': est, 'n': len(d),
            'indice_RH_coef': round(m_rh.params['indice_veg'], 2), 'indice_RH_p': round(m_rh.pvalues['indice_veg'], 4),
            'RH_PM10_coef': round(m_pm.params['RH'], 3), 'RH_PM10_p': round(m_pm.pvalues['RH'], 4),
            'indice_PM10_coef': round(m_pm.params['indice_veg'], 2), 'indice_PM10_p': round(m_pm.pvalues['indice_veg'], 4),
            'R2': round(m_pm.rsquared, 3)
        })
print(pd.DataFrame(resultados_hac).to_string(index=False))

indice estacion  n  indice_RH_coef  indice_RH_p  RH_PM10_coef  RH_PM10_p  indice_PM10_coef  indice_PM10_p    R2
  NDVI      NE2 60           -3.28       0.9554        -0.567     0.0354            197.05         0.0792 0.458
  NDVI      NE3 60            5.08       0.4156        -0.471     0.0000              6.61         0.4760 0.531
   EVI      NE2 60          108.23       0.0016        -0.805     0.0002            182.58         0.0163 0.458
   EVI      NE3 60           63.05       0.0001        -0.609     0.0006             33.99         0.1414 0.549


In [97]:
# Corrección de las BETAS (no solo de la inferencia)
resultados_beta = []
for indice in ['ndvi', 'evi']:
    for est in ['NE2', 'NE3']:
        d = construir(est, indice)
        dummies = pd.get_dummies(d.mesnum, prefix='m', drop_first=True).astype(float)
        exog = pd.concat([d[['indice_veg', 'RH']].reset_index(drop=True), dummies.reset_index(drop=True)], axis=1)
        Xc = sm.add_constant(exog)
        y = d['PM10'].astype(float)

        m_ols = sm.OLS(y, Xc).fit()
        m_hac = sm.OLS(y, Xc).fit(cov_type='HAC', cov_kwds={'maxlags': 4})
        m_pw = GLSAR(y, Xc, rho=1).iterative_fit(maxiter=20)
        rho_pw = m_pw.model.rho[0] if hasattr(m_pw.model.rho, '__len__') else m_pw.model.rho
        m_arimax = ARIMA(y, order=(1, 0, 0), exog=exog).fit(method_kwargs={'maxiter': 500})
        phi = m_arimax.params['ar.L1']

        adf_p = adfuller(y)[1]
        resid = m_arimax.resid
        lb_p = acorr_ljungbox(resid, lags=[12], return_df=True).lb_pvalue.iloc[0]
        jb_p = jarque_bera(resid)[1]
        arch_p = het_arch(resid)[1]

        resultados_beta.append({
            'indice': indice.upper(), 'estacion': est,
            'beta_OLS_HAC': round(m_hac.params['indice_veg'], 2), 'p_OLS_HAC': round(m_hac.pvalues['indice_veg'], 4),
            'beta_PraisWinsten': round(m_pw.params['indice_veg'], 2), 'p_PW': round(m_pw.pvalues['indice_veg'], 4), 'rho': round(rho_pw, 3),
            'beta_ARIMAX': round(m_arimax.params['indice_veg'], 2), 'p_ARIMAX': round(m_arimax.pvalues['indice_veg'], 4), 'phi': round(phi, 3),
            'ADF_p': round(adf_p, 4), 'LjungBox_p': round(lb_p, 4), 'JarqueBera_p': round(jb_p, 4), 'ARCH_p': round(arch_p, 4),
            'AR_estable': abs(phi) < 1
        })
print(pd.DataFrame(resultados_beta).to_string(index=False))

indice estacion  beta_OLS_HAC  p_OLS_HAC  beta_PraisWinsten   p_PW   rho  beta_ARIMAX  p_ARIMAX   phi  ADF_p  LjungBox_p  JarqueBera_p  ARCH_p  AR_estable
  NDVI      NE2        197.05     0.0792              74.82 0.2860 0.721        73.72    0.4335 0.784 0.0801      0.1534        0.5054  0.5807        True
  NDVI      NE3          6.61     0.4760               4.54 0.6858 0.394         4.88    0.7473 0.399 0.0000      0.3799        0.6015  0.9820        True
   EVI      NE2        182.58     0.0163              59.74 0.2599 0.723        57.49    0.3415 0.785 0.0801      0.0921        0.3820  0.4863        True
   EVI      NE3         33.99     0.1414              21.54 0.3267 0.372        21.46    0.2716 0.375 0.0000      0.2136        0.6320  0.9937        True


In [98]:
# Función diagnosticos_residuos(): batería de pruebas sobre un vector de residuos
def diagnosticos_residuos(resid, X_para_bp=None, lags=(6, 12)):
    """Corre la batería de pruebas sobre un vector de residuos y regresa un dict."""
    lb = acorr_ljungbox(resid, lags=list(lags), return_df=True)
    jb_stat, jb_p, skew, kurt = jarque_bera(resid)
    arch_stat, arch_p, _, _ = het_arch(resid)
    dw = durbin_watson(resid)
    out = {
        f'LjungBox_p_lag{lags[0]}': round(lb.lb_pvalue.iloc[0], 4),
        f'LjungBox_p_lag{lags[1]}': round(lb.lb_pvalue.iloc[1], 4),
        'DurbinWatson': round(dw, 3),
        'JarqueBera_p': round(jb_p, 4),
        'ARCH_p': round(arch_p, 4),
    }
    if X_para_bp is not None:
        bp_stat, bp_p, _, _ = het_breuschpagan(resid, X_para_bp)
        out['BreuschPagan_p'] = round(bp_p, 4)
    return out

In [99]:
# Función ajustar_y_diagnosticar(): compara autocorrelación antes (OLS) y después (Prais-Winsten, ARIMAX)
def ajustar_y_diagnosticar(y, exog_df, nombre_ecuacion):
    """Ajusta OLS, Prais-Winsten y ARIMAX sobre la misma ecuación, y compara
    la autocorrelación de los residuos ANTES (OLS) y DESPUÉS (PW, ARIMAX)."""
    Xc = sm.add_constant(exog_df)
    y = y.astype(float)

    m_ols = sm.OLS(y, Xc).fit()
    diag_ols = diagnosticos_residuos(m_ols.resid, X_para_bp=Xc)

    m_pw = GLSAR(y, Xc, rho=1).iterative_fit(maxiter=20)
    rho_pw = m_pw.model.rho[0] if hasattr(m_pw.model.rho, '__len__') else m_pw.model.rho
    resid_pw = m_pw.model.wendog - m_pw.model.wexog @ m_pw.params
    diag_pw = diagnosticos_residuos(resid_pw)

    m_arimax = ARIMA(y, order=(1, 0, 0), exog=exog_df).fit(method_kwargs={'maxiter': 500})
    phi = m_arimax.params['ar.L1']
    adf_p = adfuller(y)[1]
    diag_arimax = diagnosticos_residuos(m_arimax.resid)
    diag_arimax['phi'] = round(phi, 3)
    diag_arimax['AR_estable(|phi|<1)'] = abs(phi) < 1
    diag_arimax['ADF_y_p'] = round(adf_p, 4)

    print(f"\n{'='*78}\n{nombre_ecuacion}\n{'='*78}")
    print(f"{'Diagnóstico':<26}{'OLS (antes)':>15}{'Prais-Winsten':>17}{'ARIMAX':>15}")
    claves_comunes = ['LjungBox_p_lag6', 'LjungBox_p_lag12', 'DurbinWatson', 'JarqueBera_p', 'ARCH_p']
    for k in claves_comunes:
        print(f"{k:<26}{diag_ols.get(k,''):>15}{diag_pw.get(k,''):>17}{diag_arimax.get(k,''):>15}")
    print(f"\nRho estimado (Prais-Winsten) = {rho_pw:.3f}")
    print(f"Phi estimado (ARIMAX)        = {phi:.3f}  |  estable: {abs(phi) < 1}")
    print(f"ADF de la variable dependiente (estacionariedad, previo al modelo): p={adf_p:.4f}")

    return dict(ols=diag_ols, pw=diag_pw, arimax=diag_arimax, rho=rho_pw, phi=phi)

In [100]:
# Ejecuta el diagnóstico para las 4 combinaciones, en las dos ecuaciones de la mediación
resumen = []
for indice in ['ndvi', 'evi']:
    for est in ['NE2', 'NE3']:
        d = construir(est, indice)
        dummies = pd.get_dummies(d.mesnum, prefix='m', drop_first=True).astype(float)

        exog_a = pd.concat([d[['indice_veg']].reset_index(drop=True), dummies.reset_index(drop=True)], axis=1)
        diag_a = ajustar_y_diagnosticar(d['RH'], exog_a, f"{indice.upper()} - {est} - Ecuación a (índice -> RH)")

        exog_b = pd.concat([d[['indice_veg', 'RH']].reset_index(drop=True), dummies.reset_index(drop=True)], axis=1)
        diag_b = ajustar_y_diagnosticar(d['PM10'], exog_b, f"{indice.upper()} - {est} - Ecuación b (índice+RH -> PM10)")

        resumen.append({
            'indice': indice.upper(), 'estacion': est,
            'a: LB_p12 OLS': diag_a['ols']['LjungBox_p_lag12'],
            'a: LB_p12 ARIMAX': diag_a['arimax']['LjungBox_p_lag12'],
            'a: mejora': diag_a['arimax']['LjungBox_p_lag12'] > diag_a['ols']['LjungBox_p_lag12'],
            'b: LB_p12 OLS': diag_b['ols']['LjungBox_p_lag12'],
            'b: LB_p12 ARIMAX': diag_b['arimax']['LjungBox_p_lag12'],
            'b: mejora': diag_b['arimax']['LjungBox_p_lag12'] > diag_b['ols']['LjungBox_p_lag12'],
        })


NDVI - NE2 - Ecuación a (índice -> RH)
Diagnóstico                   OLS (antes)    Prais-Winsten         ARIMAX
LjungBox_p_lag6                    0.0693           0.2739         0.2554
LjungBox_p_lag12                   0.0081           0.1635          0.134
DurbinWatson                        1.491            1.871           1.86
JarqueBera_p                         0.39           0.3671         0.3657
ARCH_p                              0.505           0.7069         0.6968

Rho estimado (Prais-Winsten) = 0.260
Phi estimado (ARIMAX)        = 0.254  |  estable: True
ADF de la variable dependiente (estacionariedad, previo al modelo): p=0.0022

NDVI - NE2 - Ecuación b (índice+RH -> PM10)
Diagnóstico                   OLS (antes)    Prais-Winsten         ARIMAX
LjungBox_p_lag6                       0.0           0.2487         0.3596
LjungBox_p_lag12                      0.0           0.1241         0.1534
DurbinWatson                        0.543            2.218          2.319
Jarqu

In [101]:
# Resumen: ¿mejoró el Ljung-Box (lag 12) al pasar de OLS a ARIMAX?
print("\n\n" + "=" * 78)
print("RESUMEN — ¿mejoró el Ljung-Box (lag 12) al pasar de OLS a ARIMAX?")
print("=" * 78)
print(pd.DataFrame(resumen).to_string(index=False))



RESUMEN — ¿mejoró el Ljung-Box (lag 12) al pasar de OLS a ARIMAX?
indice estacion  a: LB_p12 OLS  a: LB_p12 ARIMAX  a: mejora  b: LB_p12 OLS  b: LB_p12 ARIMAX  b: mejora
  NDVI      NE2         0.0081            0.1340       True          0.000            0.1534       True
  NDVI      NE3         0.0215            0.2933       True          0.002            0.3799       True
   EVI      NE2         0.0124            0.0912       True          0.000            0.0921       True
   EVI      NE3         0.2285            0.5627       True          0.006            0.2136       True


#### Intervalos de Confianza con Bootstrap + Prais-Winsten

In [102]:
# Función bloque_indices(): bootstrap de bloques móviles
def bloque_indices(n, block_length, rng):
    """Bootstrap de bloques moviles: selecciona bloques de 'block_length'
    observaciones consecutivas (preservando su orden temporal interno) y los
    concatena hasta reconstruir una muestra del mismo tamaño n."""
    n_bloques = int(np.ceil(n / block_length))
    inicios = rng.integers(0, n - block_length + 1, size=n_bloques)
    idx = np.concatenate([np.arange(s, s + block_length) for s in inicios])
    return idx[:n]

In [103]:
# Función coef_ab_pw(): coeficientes a y b vía Prais-Winsten (GLSAR)
def coef_ab_pw(d_muestra):
    """Ajusta las dos ecuaciones de la mediacion con Prais-Winsten (GLSAR),
    el mismo estimador de beta corregida que se reporta en el Paso 2b, para
    que el bootstrap sea coherente con el resto del analisis."""
    dummies = pd.get_dummies(d_muestra.mesnum, prefix='m', drop_first=True).astype(float)

    X_a = sm.add_constant(pd.concat([d_muestra[['indice_veg']].reset_index(drop=True),
                                       dummies.reset_index(drop=True)], axis=1))
    a = GLSAR(d_muestra['RH'].reset_index(drop=True), X_a, rho=1).iterative_fit(maxiter=10).params['indice_veg']

    X_b = sm.add_constant(pd.concat([d_muestra[['indice_veg', 'RH']].reset_index(drop=True),
                                       dummies.reset_index(drop=True)], axis=1))
    b = GLSAR(d_muestra['PM10'].reset_index(drop=True), X_b, rho=1).iterative_fit(maxiter=10).params['RH']
    return a, b

In [104]:
# Función bootstrap_efecto_indirecto(): IC95% del efecto indirecto por bootstrap de bloques
def bootstrap_efecto_indirecto(d, n_boot=1000, block_length=6, seed=42):
    rng = np.random.default_rng(seed)
    n = len(d)
    a_obs, b_obs = coef_ab_pw(d)
    indirecto_obs = a_obs * b_obs

    dist_indirecto, fallos = [], 0
    for _ in range(n_boot):
        idx = bloque_indices(n, block_length, rng)
        d_boot = d.iloc[idx].reset_index(drop=True)
        try:
            a_b, b_b = coef_ab_pw(d_boot)
            if np.isfinite(a_b) and np.isfinite(b_b):
                dist_indirecto.append(a_b * b_b)
            else:
                fallos += 1
        except Exception:
            fallos += 1

    dist_indirecto = np.array(dist_indirecto)
    ci_low, ci_high = np.percentile(dist_indirecto, [2.5, 97.5])
    p_boot = 2 * min((dist_indirecto >= 0).mean(), (dist_indirecto <= 0).mean())
    return {
        'a_obs': a_obs, 'b_obs': b_obs, 'indirecto_obs': indirecto_obs,
        'n_boot_exitosos': len(dist_indirecto), 'fallos': fallos,
        'IC95_low': ci_low, 'IC95_high': ci_high, 'p_bootstrap': p_boot,
        'significativo': not (ci_low < 0 < ci_high)
    }

In [105]:
# Correr el bootstrap para las 4 combinaciones (reutiliza construir() ya definida)
resultados_boot = []
for indice in ['ndvi', 'evi']:
    for est in ['NE2', 'NE3']:
        d = construir(est, indice)
        r = bootstrap_efecto_indirecto(d, n_boot=1000, block_length=6)
        r['indice'] = indice.upper(); r['estacion'] = est
        resultados_boot.append(r)

tabla_boot = pd.DataFrame(resultados_boot)
print(tabla_boot[['indice', 'estacion', 'a_obs', 'b_obs', 'indirecto_obs',
                   'IC95_low', 'IC95_high', 'p_bootstrap', 'significativo']].to_string(index=False))

indice estacion      a_obs     b_obs  indirecto_obs    IC95_low  IC95_high  p_bootstrap  significativo
  NDVI      NE2 -13.395813 -0.674856       9.040250 -108.764646  84.728603        0.932          False
  NDVI      NE3   1.587672 -0.623480      -0.989883  -12.994325   6.316808        0.564          False
   EVI      NE2  97.132284 -0.740702     -71.946028 -142.258521 -24.235008        0.020           True
   EVI      NE3  58.055068 -0.697767     -40.508923  -94.105678 -13.315358        0.000           True


#### Intervalo de confianza: Monte Carlo

In [107]:
# Función monte_carlo_efecto_indirecto(): IC95% del efecto indirecto por simulación Monte Carlo
def monte_carlo_efecto_indirecto(d, n_sim=20000, seed=42, maxiter=500):
    """Ajusta ARIMAX UNA vez para cada ecuacion (a y b), y simula la
    distribucion del producto a*b a partir de sus normales asintoticas."""
    dummies = pd.get_dummies(d.mesnum, prefix='m', drop_first=True).astype(float)
    rng = np.random.default_rng(seed)

    exog_a = pd.concat([d[['indice_veg']].reset_index(drop=True), dummies.reset_index(drop=True)], axis=1)
    m_a = ARIMA(d['RH'].astype(float), order=(1, 0, 0), exog=exog_a).fit(method_kwargs={'maxiter': maxiter})
    a_hat, se_a = m_a.params['indice_veg'], m_a.bse['indice_veg']

    exog_b = pd.concat([d[['indice_veg', 'RH']].reset_index(drop=True), dummies.reset_index(drop=True)], axis=1)
    m_b = ARIMA(d['PM10'].astype(float), order=(1, 0, 0), exog=exog_b).fit(method_kwargs={'maxiter': maxiter})
    b_hat, se_b = m_b.params['RH'], m_b.bse['RH']

    a_sim = rng.normal(a_hat, se_a, n_sim)
    b_sim = rng.normal(b_hat, se_b, n_sim)
    indirecto_sim = a_sim * b_sim

    ci_low, ci_high = np.percentile(indirecto_sim, [2.5, 97.5])
    p_mc = 2 * min((indirecto_sim >= 0).mean(), (indirecto_sim <= 0).mean())

    from scipy.stats import skew
    sesgo = skew(indirecto_sim)

    return {
        'a_obs': a_hat, 'b_obs': b_hat, 'indirecto_obs': a_hat * b_hat,
        'IC95_low': ci_low, 'IC95_high': ci_high, 'p_montecarlo': p_mc,
        'significativo': not (ci_low < 0 < ci_high),
        'sesgo_distribucion_indirecto': round(sesgo, 3)
    }

In [111]:
# Correr para las 4 combinaciones (reutiliza construir() ya definida)
resultados_mc = []
for indice in ['ndvi', 'evi']:
    for est in ['NE2', 'NE3']:
        d = construir(est, indice)
        r = monte_carlo_efecto_indirecto(d, n_sim=20000)
        r['indice'] = indice.upper(); r['estacion'] = est
        resultados_mc.append(r)
        print(f"{indice.upper()} {est}: indirecto={r['indirecto_obs']:.2f}, "
              f"IC95%=[{r['IC95_low']:.2f}, {r['IC95_high']:.2f}], "
              f"p={r['p_montecarlo']:.4f}, sig={r['significativo']}, "
              f"sesgo={r['sesgo_distribucion_indirecto']}")

NDVI NE2: indirecto=11.30, IC95%=[-56.19, 80.26], p=0.7356, sig=False, sesgo=0.077
NDVI NE3: indirecto=-1.15, IC95%=[-18.72, 15.61], p=0.8866, sig=False, sesgo=-0.069
EVI NE2: indirecto=-71.31, IC95%=[-149.58, -7.04], p=0.0288, sig=True, sesgo=-0.368
EVI NE3: indirecto=-40.12, IC95%=[-71.24, -15.76], p=0.0003, sig=True, sesgo=-0.498


In [109]:
# Mostrar tabla resumen de Monte Carlo
print()
print(pd.DataFrame(resultados_mc)[['indice','estacion','a_obs','b_obs','indirecto_obs',
                                     'IC95_low','IC95_high','p_montecarlo','significativo',
                                     'sesgo_distribucion_indirecto']].to_string(index=False))


indice estacion      a_obs     b_obs  indirecto_obs    IC95_low  IC95_high  p_montecarlo  significativo  sesgo_distribucion_indirecto
  NDVI      NE2 -16.450039 -0.687174      11.304046  -56.192154  80.261854        0.7356          False                         0.077
  NDVI      NE3   1.842453 -0.625536      -1.152521  -18.723555  15.606321        0.8866          False                        -0.069
   EVI      NE2  94.943486 -0.751105     -71.312564 -149.580837  -7.041249        0.0288           True                        -0.368
   EVI      NE3  57.825725 -0.693778     -40.118195  -71.243991 -15.762793        0.0003           True                        -0.498
